# 第 2 周练习 —— Code Explainer Chatbot（代码讲解聊天机器人）

## 练习目标（理念）

把第 1 周技术问答做成**完整 Gradio 原型**，并加上第 2 周的进阶点：

- **Gradio `Blocks` + `ChatInterface`**：可布局的聊天 UI
- **流式回复**：Claude 与 Llama 两条路径都尽量边生成边显示
- **专家 System Prompt**：资深 Python 工程师 + 分步讲解代码
- **模型切换**：Claude（Anthropic SDK）↔ Llama 3.2（Ollama）
- **Tool Calling**：`lookup_python_docs` 查官方文档链接

## 和本课第 2 周的关系

| 本课概念 | 本笔记本里你会看到 |
|----------|-------------------|
| System Prompt | `SYSTEM_PROMPT` |
| Tools / tool_use | `docs_tool` + `handle_tool_calls` |
| Streaming | Anthropic `messages.stream` / Ollama `stream=True` |
| 多后端 | Anthropic 云端 + 本地 Ollama |

## 怎么跑

1. `.env` 准备 `ANTHROPIC_API_KEY`；本地需已 `ollama pull llama3.2`
2. 从上到下运行；最后一格 `ui.launch(share=True)` 打开界面
3. 粘贴代码片段，切换模型对比讲解风格


In [1]:
# ========== 导入：Anthropic、Ollama、Gradio 等 ==========

# 标准库 os：读/写环境变量（例如 ANTHROPIC_API_KEY）
import os
# 标准库 json：本练习导入备用（工具参数有时是 JSON 字符串）
import json
# load_dotenv：从 .env 加载密钥到进程环境
from dotenv import load_dotenv
# Anthropic 官方 SDK：messages.create / messages.stream，支持 tools
from anthropic import Anthropic
# ollama Python 包：调本地 Llama（chat + stream）
import ollama
# Gradio：Blocks / ChatInterface / Dropdown
import gradio as gr


In [2]:
# ========== 环境：加载 ANTHROPIC_API_KEY，必要时交互输入 ==========

# override=True：.env 覆盖已存在的同名环境变量
load_dotenv(override=True)

# 读取 Anthropic 密钥
api_key = os.getenv('ANTHROPIC_API_KEY')
# 若缺失：用 getpass 让用户在终端隐式输入，再写回 os.environ
if not api_key:
    import getpass
    # 提示文案保持英文原样
    print('ANTHROPIC_API_KEY not found. Please enter it:')
    api_key = getpass.getpass()
    os.environ['ANTHROPIC_API_KEY'] = api_key

# 创建 Anthropic 客户端（默认读环境变量 ANTHROPIC_API_KEY）
anthropic_client = Anthropic()
# 打印前缀方便确认已加载（勿泄露完整 key）
print(f'Anthropic API Key loaded (starts with {api_key[:12]}...)')


Anthropic API Key loaded (starts with sk-ant-api03...)


In [4]:
# ========== 模型选项：Claude 与本地 Llama ==========

# Anthropic 云端模型 id（字符串必须与账号可用模型一致）
MODEL_CLAUDE = 'claude-sonnet-4-6'
# 本地 Ollama 模型名（需事先 pull）
MODEL_LLAMA = 'llama3.2'
# Gradio 下拉框的可选列表
MODEL_CHOICES = [MODEL_CLAUDE, MODEL_LLAMA]


In [5]:
# ========== System Prompt：代码讲解专家（英文指令保留，改译会改行为）==========

# SYSTEM_PROMPT：传给 Anthropic 的 system=，以及 Ollama messages 里的 system 角色
SYSTEM_PROMPT = """
You are a senior Python engineer and educator.
When a user shares a code snippet or asks a technical question:

1. Identify the key constructs and patterns used.
2. Explain what the code does in plain English.
3. Walk through the execution step-by-step.
4. Highlight any potential pitfalls or best-practice improvements.

Keep your explanations clear and concise. Use markdown formatting for readability.
If you need to look up documentation for a Python built-in, use the lookup_python_docs tool.
"""


In [6]:
# ========== Tool：lookup_python_docs —— 主题 → 官方文档 URL ==========

# 本地字典：常见 Python 概念到 docs.python.org 的映射（URL 保持原样）
PYTHON_DOCS = {
    'yield': 'https://docs.python.org/3/reference/expressions.html#yield-expressions',
    'yield from': 'https://docs.python.org/3/reference/expressions.html#yield-expressions',
    'list comprehension': 'https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions',
    'dict comprehension': 'https://docs.python.org/3/tutorial/datastructures.html#dictionaries',
    'set comprehension': 'https://docs.python.org/3/tutorial/datastructures.html#sets',
    'generator': 'https://docs.python.org/3/howto/functional.html#generators',
    'decorator': 'https://docs.python.org/3/glossary.html#term-decorator',
    'lambda': 'https://docs.python.org/3/reference/expressions.html#lambda',
    'map': 'https://docs.python.org/3/library/functions.html#map',
    'filter': 'https://docs.python.org/3/library/functions.html#filter',
    'reduce': 'https://docs.python.org/3/library/functools.html#functools.reduce',
    'zip': 'https://docs.python.org/3/library/functions.html#zip',
    'enumerate': 'https://docs.python.org/3/library/functions.html#enumerate',
    'async': 'https://docs.python.org/3/library/asyncio.html',
    'await': 'https://docs.python.org/3/library/asyncio-task.html#awaitables',
    'dataclass': 'https://docs.python.org/3/library/dataclasses.html',
    'namedtuple': 'https://docs.python.org/3/library/collections.html#collections.namedtuple',
}

def lookup_python_docs(topic):
    """按主题查 PYTHON_DOCS；命中则返回 Markdown 链接文案，否则给搜索页提示。"""
    # 规范化：小写 + 去首尾空白，便于字典查找
    topic_lower = topic.lower().strip()
    url = PYTHON_DOCS.get(topic_lower)
    if url:
        # 返回给模型/用户看的说明字符串（格式保持原样）
        return f'Documentation for **{topic}**: {url}'
    return f'No specific documentation link found for "{topic}". Try searching at https://docs.python.org/3/search.html?q={topic}'

# Anthropic tool schema：name/description/input_schema 是发给模型的工具定义（英文保留）
docs_tool = {
    'name': 'lookup_python_docs',
    'description': 'Look up the official Python documentation URL for a given Python concept, built-in, or keyword.',
    'input_schema': {
        'type': 'object',
        'properties': {
            'topic': {
                'type': 'string',
                'description': 'The Python concept or keyword to look up, e.g. yield, lambda, decorator',
            },
        },
        'required': ['topic'],
    },
}
# tools 列表：传给 messages.create(..., tools=tools)
tools = [docs_tool]


In [7]:
# ========== 处理 tool_use：执行本地函数，拼回 tool_result ==========

def handle_tool_calls(message):
    """遍历 assistant 消息的 content 块；对 lookup_python_docs 调用本地函数并组装 tool_result。"""
    responses = []
    # Anthropic 一条消息的 content 可能是多个 block（text / tool_use 等）
    for content_block in message.content:
        # 只处理「调用 lookup_python_docs」的 tool_use 块
        if content_block.type == 'tool_use' and content_block.name == 'lookup_python_docs':
            # 从模型给出的 input 字典取 topic
            topic = content_block.input.get('topic', '')
            # 真正执行本地工具
            result = lookup_python_docs(topic)
            # Anthropic 要求：用 tool_result + 对应 tool_use_id 回传结果
            responses.append({
                'type': 'tool_result',
                'tool_use_id': content_block.id,
                'content': result,
            })
    return responses


In [8]:
# ========== chat 回调：按模型分流（Claude 可工具 + 流式；Llama 仅流式）==========

def chat(message, history, model_name):
    """Gradio 聊天回调：根据 model_name 走 Anthropic 或 Ollama，并流式 yield 文本。"""
    # 规整 history 为纯 role/content 列表
    history = [{'role': h['role'], 'content': h['content']} for h in history]

    if model_name == MODEL_CLAUDE:
        # --- Anthropic 路径：先非流式探测是否要调工具 ---
        messages_for_claude = history + [{'role': 'user', 'content': message}]
        
        # 第一次请求：不 stream，以便检查 stop_reason 是否为 tool_use
        response = anthropic_client.messages.create(
            model=MODEL_CLAUDE, 
            system=SYSTEM_PROMPT,
            max_tokens=1024,
            messages=messages_for_claude, 
            tools=tools
        )
        
        if response.stop_reason == 'tool_use':
            # 执行工具，把 assistant 原消息 + tool_result（包在 user content 里）追加进对话
            tool_responses = handle_tool_calls(response)
            messages_for_claude.append({'role': 'assistant', 'content': response.content})
            messages_for_claude.append({'role': 'user', 'content': tool_responses})
            
            # 工具结果回来后，再流式生成最终讲解
            with anthropic_client.messages.stream(
                model=MODEL_CLAUDE, 
                system=SYSTEM_PROMPT,
                max_tokens=1024,
                messages=messages_for_claude, 
                tools=tools
            ) as stream:
                result = ''
                # text_stream：只迭代文本增量
                for text in stream.text_stream:
                    result += text
                    yield result
        else:
            # 模型没用工具：直接取出首个 text block（本分支未再开 stream）
             yield response.content[0].text

    else:
        # --- Ollama 路径：本地流式聊天，不做 tool calling ---
        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + history + [{'role': 'user', 'content': message}]
        stream = ollama.chat(
            model=MODEL_LLAMA,
            messages=messages,
            stream=True,
        )
        result = ''
        for chunk in stream:
            # Ollama 流式块结构：chunk['message']['content']
            result += chunk['message']['content']
            yield result


In [ ]:
# ========== 启动 Gradio UI：Blocks 布局 + ChatInterface ==========

# with gr.Blocks：自定义标题区 + 模型下拉，再嵌 ChatInterface
with gr.Blocks(title='Code Explainer Chatbot') as ui:
    # Markdown 标题/说明字符串保持英文原样（UI 文案）
    gr.Markdown('##Code Explainer Chatbot')
    gr.Markdown('Paste a code snippet and get a step-by-step explanation. Switch models anytime.')

    # 一行放模型下拉；choices 来自 MODEL_CHOICES，默认 Claude
    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=MODEL_CHOICES,
            value=MODEL_CLAUDE,
            label='Select Model',
        )

    # ChatInterface：fn=chat；additional_inputs 把下拉值传给 chat 的 model_name
    chatbot = gr.ChatInterface(
        fn=chat,
        type='messages',
        additional_inputs=[model_dropdown],
    )

# share=True：尝试生成公网可分享链接（需 Gradio 支持）；本地也会开端口
ui.launch(share=True)
